# FloodNet U-Net corrected-loss GPU run

This notebook clones the existing repository at a pinned commit and calls its maintained `src.train` and `src.evaluate` modules. It does not duplicate model or training code. The corrected Dice loss excludes class-absent images from each class Dice average. Training and epoch validation use only the FloodNet `train/` and `val/` splits. The test split is reserved for one final evaluation after validation shows Road-flooded detection.

## One-time Colab setup

In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another GPU). Put the extracted FloodNet Supervised v1.0 folder in Google Drive at `MyDrive/FloodNet-Supervised_v1.0/`, with `train/`, `val/`, and `test/` folders. Upload the existing ignored checkpoint to `MyDrive/FloodNet_Checkpoints/floodnet_unet_best.pth`. The source is fetched from GitHub; no second code copy is needed. All new checkpoints, epoch history, evaluation reports, and visualizations go under the separate `MyDrive/FloodNet_CorrectedDice_Run_2026-10-21/` folder, leaving historical baseline reports untouched.

Colab GPU allocation and Drive access depend on the signed-in account. If Colab cannot allocate a GPU, do not proceed with full training on CPU.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Fetch the canonical project and verify inputs

The pinned source revision contains the corrected loss, weights-only initialization, persistent checkpoint output, a separate best-Road-flooded checkpoint, and per-class validation logging. This cell checks all three split directories and the starting checkpoint before training. It preserves Colab's CUDA-enabled PyTorch installation.

In [ ]:
from pathlib import Path
import os, subprocess, sys
import torch, torchvision

PROJECT_DIR = Path("/content/Road_WaterLogging_Project-")
REPO_URL = "https://github.com/vaishnavij1612/Road_WaterLogging_Project-.git"
PROJECT_SOURCE_COMMIT = "9b8058b6a5b6c303a1c42b0eaa40dc97bed633d6"
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "--detach", PROJECT_SOURCE_COMMIT], check=True)
os.chdir(PROJECT_DIR)

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. In Colab select Runtime > Change runtime type > GPU, then rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)

FLOODNET_DIR = Path("/content/drive/MyDrive/FloodNet-Supervised_v1.0")
BASELINE_CHECKPOINT = Path("/content/drive/MyDrive/FloodNet_Checkpoints/floodnet_unet_best.pth")
RUN_DIR = Path("/content/drive/MyDrive/FloodNet_CorrectedDice_Run_2026-10-21")
CHECKPOINT_DIR = RUN_DIR
REPORTS_DIR = RUN_DIR
VISUALIZATIONS_DIR = RUN_DIR / "visualizations"

required_dirs = [
    FLOODNET_DIR / "train/train-org-img", FLOODNET_DIR / "train/train-label-img",
    FLOODNET_DIR / "val/val-org-img", FLOODNET_DIR / "val/val-label-img",
    FLOODNET_DIR / "test/test-org-img", FLOODNET_DIR / "test/test-label-img",
]
missing = [str(path) for path in required_dirs if not path.is_dir()]
if missing:
    raise FileNotFoundError("Missing FloodNet split directories:\n" + "\n".join(missing))
if not BASELINE_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Upload the existing best checkpoint to {BASELINE_CHECKPOINT}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
VISUALIZATIONS_DIR.mkdir(parents=True, exist_ok=True)
print("Project commit:", PROJECT_SOURCE_COMMIT)
print("FloodNet root:", FLOODNET_DIR)
print("Starting checkpoint:", BASELINE_CHECKPOINT)
print("Persistent run directory:", RUN_DIR)

## Train with epoch-by-epoch validation

The first run initializes from the old best model **weights only**, resetting its optimizer/scheduler state because those Adam moments were accumulated under the faulty loss. Later reruns resume this corrected run from its persistent `last` checkpoint. Every epoch validates on `val/`, logs Road-flooded IoU plus all present per-class IoUs, and persists latest, best-mIoU, and best-Road-flooded checkpoints with the run history. To start a distinct run without overwriting this one, change `RUN_DIR` above to a new unique Drive folder.

In [ ]:
last_checkpoint = CHECKPOINT_DIR / "floodnet_unet_last.pth"
cmd = [
    sys.executable, "-m", "src.train",
    "--floodnet-dir", str(FLOODNET_DIR),
    "--epochs", "20", "--batch-size", "4", "--base-features", "32",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--reports-dir", str(REPORTS_DIR),
    "--drive-dir", str(RUN_DIR),
]
if last_checkpoint.is_file():
    cmd.extend(["--resume", str(last_checkpoint)])
else:
    cmd.extend(["--init-weights", str(BASELINE_CHECKPOINT), "--no-auto-resume"])
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
print("Persistent epoch history:", REPORTS_DIR / "training_history.json")

## Validation checkpoint selection and genuine prediction visuals

The trainer saves the epoch with the highest validation Road-flooded IoU separately from the best-mIoU checkpoint. Select that class-specific checkpoint when it exists; otherwise inspect the best-mIoU model and training history. This cell evaluates only validation, saves its JSON report to Drive, and copies genuine original/ground-truth/prediction panels to Drive. Review the validation metrics and panels before deciding whether Road-flooded detection is effective. The held-out test split is not used here.

In [ ]:
import json, shutil
from IPython.display import Image as DisplayImage, display

best_road_checkpoint = CHECKPOINT_DIR / "floodnet_unet_best_road_flooded.pth"
best_miou_checkpoint = CHECKPOINT_DIR / "floodnet_unet_best.pth"
if best_road_checkpoint.is_file():
    SELECTED_CHECKPOINT = best_road_checkpoint
else:
    SELECTED_CHECKPOINT = best_miou_checkpoint
if not SELECTED_CHECKPOINT.is_file():
    raise FileNotFoundError("Training did not produce a best checkpoint.")

checkpoint = torch.load(SELECTED_CHECKPOINT, map_location="cpu", weights_only=False)
print("Selected:", SELECTED_CHECKPOINT.name)
print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Validation Road-flooded IoU:", (checkpoint.get("val_metrics") or {}).get("road_flooded_iou"))
print("Validation per-class IoU:", json.dumps((checkpoint.get("val_metrics") or {}).get("per_class_iou", {}), indent=2))

val_report = REPORTS_DIR / "evaluation_val.json"
subprocess.run([
    sys.executable, "-m", "src.evaluate", "--split", "val",
    "--checkpoint", str(SELECTED_CHECKPOINT), "--output", str(val_report),
], cwd=PROJECT_DIR, check=True)
for source in sorted((PROJECT_DIR / "outputs/visualizations").glob("val_comparison_*.png")):
    target = VISUALIZATIONS_DIR / source.name
    shutil.copy2(source, target)
    display(DisplayImage(filename=str(target)))
print("Validation report:", val_report)

## Final test evaluation — run once only after validation review

Do not enable the test cell until the validation report and prediction panels show Road-flooded pixels in plausible road regions. It is disabled by default and refuses to run if validation Road-flooded IoU is zero. After the validation review, set `RUN_FINAL_TEST = True` and execute this cell once. It writes the held-out result to a separate Drive report and copies the genuine test comparison panels to the corrected-run folder.

In [ ]:
RUN_FINAL_TEST = False  # Enable only after reviewing effective validation Road-flooded detection.
if RUN_FINAL_TEST:
    with val_report.open("r", encoding="utf-8") as f:
        reviewed_val = json.load(f)
    if reviewed_val.get("road_flooded_iou", 0.0) <= 0.0:
        raise RuntimeError("Validation Road-flooded IoU is zero; do not evaluate the held-out test set yet.")
    test_report = REPORTS_DIR / "evaluation_test_final.json"
    subprocess.run([
        sys.executable, "-m", "src.evaluate", "--split", "test",
        "--checkpoint", str(SELECTED_CHECKPOINT), "--output", str(test_report),
    ], cwd=PROJECT_DIR, check=True)
    for source in sorted((PROJECT_DIR / "outputs/visualizations").glob("test_comparison_*.png")):
        shutil.copy2(source, VISUALIZATIONS_DIR / source.name)
    print("One-time final test report:", test_report)
else:
    print("Test set untouched. Review validation first; then set RUN_FINAL_TEST = True for its one-time final evaluation.")